# Retrofit Document Types (Non-Bank Only)

Updates ground truth `DOCUMENT_TYPE` with model predictions **only for non-bank-statement images**.
Bank statement rows are never modified.

In [ ]:
# Verify GT Override: Are All Predicted Document Types Correct?
#
# Point these at your latest run's GT CSV and results directory.
# This cell is standalone — run it independently to check for misclassifications.

from pathlib import Path

import pandas as pd

# --- CONFIGURE THESE ---
GT_CSV = Path("/efs/shared/annotations/lmm_poc_gt_20260325.csv")
RESULTS_DIR = Path("/efs/shared/annotations/output/batch")

# Auto-find latest results CSV
results_files = sorted(
    f for f in RESULTS_DIR.glob("batch_*_results*.csv") if "summary" not in f.name
)
if not results_files:
    results_files = sorted(RESULTS_DIR.glob("batch_results_*.csv"))
assert results_files, f"No batch results CSV found in {RESULTS_DIR}"
RESULTS_CSV = results_files[-1]

gt = pd.read_csv(GT_CSV, dtype=str)
results = pd.read_csv(RESULTS_CSV, dtype=str)
print(f"GT:      {GT_CSV} ({len(gt)} rows)")
print(f"Results: {RESULTS_CSV} ({len(results)} rows)")

# Find image columns
IMG_COLS = ["image_name", "image_file", "filename", "file"]
gt_img = next(c for c in IMG_COLS if c in gt.columns)
res_img = next(c for c in IMG_COLS if c in results.columns)

# Build predicted type map
pred_map = {
    Path(k).stem: v
    for k, v in results.set_index(res_img)["document_type"].to_dict().items()
}

# Compare
rows = []
for _, row in gt.iterrows():
    stem = Path(row[gt_img]).stem
    gt_type = str(row.get("DOCUMENT_TYPE", "")).strip().upper()
    pred_type = str(pred_map.get(stem, "MISSING")).strip().upper()
    rows.append({"image": stem, "gt_type": gt_type, "pred_type": pred_type})

cmp = pd.DataFrame(rows)
cmp["match"] = cmp["gt_type"] == cmp["pred_type"]

# Confusion matrix
print("\n=== Confusion Matrix (GT rows x Predicted cols) ===\n")
display(pd.crosstab(cmp["gt_type"], cmp["pred_type"], margins=True))

# Mismatches
mismatches = cmp[~cmp["match"]]
print(f"\n=== Misclassifications: {len(mismatches)} / {len(cmp)} ===")
if len(mismatches) > 0:
    display(mismatches)
else:
    print("None — model classified every document correctly.")

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# --- Configure paths ---
GROUND_TRUTH_CSV = Path("../evaluation_data/bank/ground_truth_bank.csv")
RESULTS_DIR = Path("../evaluation_data/output/batch")

# Auto-find latest results CSV (match both naming conventions)
results_files = sorted(
    f for f in RESULTS_DIR.glob("batch_*_results*.csv") if "summary" not in f.name
)
if not results_files:
    # Fallback: try legacy naming
    results_files = sorted(RESULTS_DIR.glob("batch_results_*.csv"))
assert results_files, f"No batch results CSV found in {RESULTS_DIR}"
RESULTS_CSV = results_files[-1]
print(f"Ground truth: {GROUND_TRUTH_CSV}")
print(f"Results CSV:  {RESULTS_CSV}")

In [ ]:
# Load both CSVs
gt = pd.read_csv(GROUND_TRUTH_CSV, dtype=str)
results = pd.read_csv(RESULTS_CSV, dtype=str)

print(f"Ground truth columns: {list(gt.columns)}")
print(f"Results columns:      {list(results.columns)}")

# Find image column in each CSV
IMAGE_COL_CANDIDATES = ["image_name", "image_file", "filename", "file"]
gt_image_col = next(c for c in IMAGE_COL_CANDIDATES if c in gt.columns)
results_image_col = next(c for c in IMAGE_COL_CANDIDATES if c in results.columns)
print(f"\nGT image column:      {gt_image_col}")
print(f"Results image column: {results_image_col}")
print(f"Ground truth: {len(gt)} rows")
print(f"Results:      {len(results)} rows")

# Show existing DOCUMENT_TYPE distribution
if "DOCUMENT_TYPE" in gt.columns:
    print("\nExisting ground truth DOCUMENT_TYPE:")
    print(gt["DOCUMENT_TYPE"].value_counts(dropna=False))
else:
    print("\nNo DOCUMENT_TYPE column in ground truth yet")

In [ ]:
# DEBUG: compare image_name values between CSVs (using stem to strip extensions)
gt_names = set(gt[gt_image_col].apply(lambda x: Path(x).stem))
results_names = set(results[results_image_col].apply(lambda x: Path(x).stem))

print(f"GT unique stems:      {len(gt_names)}")
print(f"Results unique stems: {len(results_names)}")
print(f"Intersection:         {len(gt_names & results_names)}")
print(f"In GT only:           {len(gt_names - results_names)}")
print(f"In results only:      {len(results_names - gt_names)}")

print("\n--- Sample GT image_name values ---")
for v in sorted(gt[gt_image_col].head(5)):
    print(f"  {v!r}")

print("\n--- Sample results image_name values ---")
for v in sorted(results[results_image_col].head(5)):
    print(f"  {v!r}")

## Classification Impact Analysis

Compare GT vs predicted document types to assess misclassification impact.
- **Receipt ↔ Invoice**: Low impact — similar extraction fields, same pipeline
- **Receipt/Invoice → Bank Statement**: HIGH impact — wrong pipeline (multi-turn bank extractor)
- **Bank Statement → Receipt/Invoice**: HIGH impact — bank fields lost entirely

In [ ]:
# Build comparison: GT doc type vs predicted doc type
pred_map = {
    Path(k).stem: v
    for k, v in results.set_index(results_image_col)["document_type"].to_dict().items()
}

comparison = []
for _, row in gt.iterrows():
    stem = Path(row[gt_image_col]).stem
    gt_type = str(row.get("DOCUMENT_TYPE", "")).strip().upper()
    pred_type = str(pred_map.get(stem, "")).strip().upper()
    comparison.append({"image": stem, "gt_type": gt_type, "pred_type": pred_type})

cmp = pd.DataFrame(comparison)
cmp["match"] = cmp["gt_type"] == cmp["pred_type"]

# Confusion matrix
print("=== Confusion Matrix (GT rows vs Predicted columns) ===\n")
confusion = pd.crosstab(cmp["gt_type"], cmp["pred_type"], margins=True)
display(confusion)

# Highlight cross-pipeline misclassifications
bank_set = {"BANK_STATEMENT"}
non_bank_set = {"RECEIPT", "INVOICE"}

cross_pipeline = cmp[
    ((cmp["gt_type"].isin(non_bank_set)) & (cmp["pred_type"].isin(bank_set)))
    | ((cmp["gt_type"].isin(bank_set)) & (cmp["pred_type"].isin(non_bank_set)))
]

print(f"\n=== Cross-Pipeline Misclassifications: {len(cross_pipeline)} ===")
if len(cross_pipeline) > 0:
    print("These images would be routed to the WRONG extraction pipeline!\n")
    display(cross_pipeline)
else:
    print("None found — all bank vs non-bank classifications are correct.")

# Receipt <-> Invoice (low impact)
receipt_invoice_swap = cmp[
    ((cmp["gt_type"] == "RECEIPT") & (cmp["pred_type"] == "INVOICE"))
    | ((cmp["gt_type"] == "INVOICE") & (cmp["pred_type"] == "RECEIPT"))
]
print(f"\n=== Receipt ↔ Invoice Swaps (low impact): {len(receipt_invoice_swap)} ===")
if len(receipt_invoice_swap) > 0:
    display(receipt_invoice_swap)

# Summary
total = len(cmp)
matched = cmp["match"].sum()
print("\n=== Summary ===")
print(f"Total images:              {total}")
print(f"Correct classification:    {matched} ({matched / total:.1%})")
print(f"Receipt/Invoice swaps:     {len(receipt_invoice_swap)} (low impact)")
print(f"Cross-pipeline errors:     {len(cross_pipeline)} (HIGH impact)")

In [ ]:
# Build predicted type map keyed by stem (no extension) for cross-format matching
type_map = {
    Path(k).stem: v
    for k, v in results.set_index(results_image_col)["document_type"].to_dict().items()
}

# Update only non-bank rows, tracking each remapping
updated = 0
skipped_bank = 0
no_prediction = 0
remappings = []  # (image, old_type, new_type)

if "DOCUMENT_TYPE" not in gt.columns:
    gt["DOCUMENT_TYPE"] = pd.NA

for idx, row in gt.iterrows():
    image_name = Path(row[gt_image_col]).stem  # normalize to stem (no extension)
    existing_type = str(row.get("DOCUMENT_TYPE", "")).strip().upper()
    predicted_type = type_map.get(image_name)

    # Never touch bank statements
    if existing_type == "BANK_STATEMENT":
        skipped_bank += 1
        continue

    if predicted_type:
        old_type = row.get("DOCUMENT_TYPE", "")
        gt.loc[idx, "DOCUMENT_TYPE"] = predicted_type
        remappings.append((image_name, old_type, predicted_type))
        updated += 1
    else:
        no_prediction += 1

print(f"Updated:        {updated}")
print(f"Skipped (bank): {skipped_bank}")
print(f"No prediction:  {no_prediction}")

# Show all remappings for review
if remappings:
    print(f"\n--- Remappings ({len(remappings)}) ---")
    remap_df = pd.DataFrame(remappings, columns=["image", "old_type", "new_type"])
    remap_df["changed"] = remap_df["old_type"] != remap_df["new_type"]
    display(remap_df)

In [ ]:
# Final distribution
print("Updated DOCUMENT_TYPE distribution:")
print(gt["DOCUMENT_TYPE"].value_counts(dropna=False))
print()
gt[[gt_image_col, "DOCUMENT_TYPE"]]

In [ ]:
# Save — overwrites original ground truth
gt.to_csv(GROUND_TRUTH_CSV, index=False)
print(f"Saved: {GROUND_TRUTH_CSV}")

## Review Misclassified Bank Statements

These 3 images are bank statements that the model classified as receipts.
Display each image alongside its GT field values so we can verify the receipt fields are correct.

Edit the GT values inline if needed, then run the save cell.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image

# --- CONFIGURE ---
GT_CSV = Path("/efs/shared/annotations/lmm_poc_gt_20260325.csv")
IMAGE_DIR = Path("/efs/shared/annotations/annotation_images")

MISCLASSIFIED = [
    "1351904028_1_1",
    "1353775322_1_58",
    "1364080459_4_2",
]

RECEIPT_FIELDS = [
    "DOCUMENT_TYPE",
    "BUSINESS_ABN",
    "SUPPLIER_NAME",
    "BUSINESS_ADDRESS",
    "PAYER_NAME",
    "PAYER_ADDRESS",
    "INVOICE_DATE",
    "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES",
    "LINE_ITEM_PRICES",
    "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED",
    "GST_AMOUNT",
    "TOTAL_AMOUNT",
]

# Load GT
gt = pd.read_csv(GT_CSV, dtype=str)
IMG_COLS = ["image_name", "image_file", "filename", "file"]
gt_img_col = next(c for c in IMG_COLS if c in gt.columns)

# Display each image with its GT values
for stem in MISCLASSIFIED:
    # Find the GT row
    mask = gt[gt_img_col].apply(lambda x, s=stem: Path(x).stem == s)
    if not mask.any():
        print(f"\n{'=' * 60}")
        print(f"WARNING: {stem} not found in GT CSV")
        continue

    row = gt.loc[mask].iloc[0]

    # Find the image file
    matches = list(IMAGE_DIR.glob(f"{stem}.*"))
    print(f"\n{'=' * 60}")
    print(f"IMAGE: {stem}")
    print(f"{'=' * 60}")

    if matches:
        img = Image.open(matches[0])
        # Resize for display
        max_width = 600
        if img.width > max_width:
            ratio = max_width / img.width
            img = img.resize((max_width, int(img.height * ratio)))
        display(img)
    else:
        print(f"  (image file not found in {IMAGE_DIR})")

    # Show GT fields
    print(f"\nGT fields for {stem}:")
    fields_present = [f for f in RECEIPT_FIELDS if f in gt.columns]
    for field in fields_present:
        val = row.get(field, "")
        print(f"  {field:30s} = {val}")

In [ ]:
import json

# Load current GT values for the 3 misclassified images as an editable dict.
# Review the images above, then edit values directly in the `corrections` dict.
# Run this cell to apply changes and save.

corrections = {}
for stem in MISCLASSIFIED:
    mask = gt[gt_img_col].apply(lambda x, s=stem: Path(x).stem == s)
    if not mask.any():
        print(f"WARNING: {stem} not found in GT")
        continue
    row = gt.loc[mask].iloc[0]
    fields_present = [f for f in RECEIPT_FIELDS if f in gt.columns]
    corrections[stem] = {f: str(row.get(f, "")) for f in fields_present}

# Pretty-print as editable Python dict
print("corrections = {")
for stem, fields in corrections.items():
    print(f'    "{stem}": {{')
    for field, value in fields.items():
        print(f'        "{field}": {json.dumps(value)},')
    print("    },")
print("}")
print("\n# Copy the above, paste into the next cell, edit values, then run it.")

In [ ]:
# Paste the corrections dict from the cell above, edit values, then run this cell.
# Only fields you change will be updated — unchanged fields are skipped.

corrections = {
    # Paste here and edit
}

# Apply corrections (only saves fields that differ from current GT)
changed = 0
for stem, fields in corrections.items():
    if not fields:
        continue
    mask = gt[gt_img_col].apply(lambda x, s=stem: Path(x).stem == s)
    if not mask.any():
        print(f"WARNING: {stem} not found in GT")
        continue
    idx = gt.loc[mask].index[0]
    for field, value in fields.items():
        old = str(gt.at[idx, field]) if field in gt.columns else "(missing)"
        if old != value:
            gt.at[idx, field] = value
            print(f"  {stem}: {field} = {old!r} -> {value!r}")
            changed += 1

print(f"\nChanged {changed} field(s)")

if changed:
    gt.to_csv(GT_CSV, index=False)
    print(f"Saved: {GT_CSV}")
else:
    print("No changes detected — GT not modified.")